# model_14 - YOL MODELIMimari `deneme2/DENKLEM.md`, olcu `olcme_14`:    DOGRULUK      BICIM   kalip / ek / tip        cumle OGRETILDIGI GIBI mi kuruldu      BILGI   OGRETILEN / CIKARIM     cumledeki bilgi dogru mu**Kural 8 - butun kosular ARKA PLANDA.** Uzun is bir iplikte baslar,hucre ANINDA doner. Durum icin `R RAPOR` hucresi: CPU, aninda doner,her zaman guvenli, uc kosunun da tam logunu basar.**Korpus burada URETILMEZ.** Birim akisi Drive'dan okunur(`MyDrive/model_14/birim_14.npz`, ~9 MB). Grafi `veri_kur` tohumdan1 sn'de kuruyor. Karakter yolu (jeton, 512'lik paketleme, 164 MBonbellek) Colab'da hic calismaz - model_14 karakter gormuyor.**Tek tohum (t0).** Mimarinin ogrenilen kismi donmeler; sabit noktabulutu tohumdan geliyor ama donmeler hangi buluta verilirse ona uyumsagliyor. Olumsuz sonucta t1/t2 eklenir (ISIMLENDIRME.md).

In [ ]:
# 0 GPU KAPISI  |  GPU  |  tekrar: GUVENLI# CLAUDE.md kural 2: GPU'yu kullanacak hucre GPU'yu KENDI ICINDE sorar.# Ayri bir "GPU var mi" hucresi hucre sirasina bagli bir kuraldir --# insan hatirlarsa calisir.import os, sys, time, json, threading, subprocessimport torchassert torch.cuda.is_available(), "GPU YOK -- Runtime > Change runtime type"_bos = torch.cuda.mem_get_info()[0] / 1e9assert _bos > 2.0, "GPU'da sadece %.1f GB bos" % _bosprint("GPU kapisi GECTI: %s  bos %.1f GB" % (torch.cuda.get_device_name(0), _bos))print("torch", torch.__version__)

In [ ]:
# 1 DEPO + DRIVE  |  CPU  |  tekrar: GUVENLI  |  ~15 snfrom google.colab import drivedrive.mount("/content/drive")DEPO = "https://github.com/sekerahmet/sekerai.git"KOD = "/content/kod"subprocess.run(["rm", "-rf", KOD], check=True)subprocess.run(["git", "clone", "-q", "--depth", "1", DEPO, KOD], check=True)KOL = KOD + "/deneme2/model_14"if KOL not in sys.path:    sys.path.insert(0, KOL)# !! MODUL ONBELLEGI. rm -rf + clone dosyayi degistirir ama `import`# ONCEKI modulu dondurur -- model_13'te bir kez bunun kurbani olundu.for _m in [m for m in list(sys.modules) if m.endswith("_14")]:    del sys.modules[_m]CIKTI = "/content/drive/MyDrive/model_14/t0"BIRIM_YOL = "/content/drive/MyDrive/model_14/birim_14.npz"os.makedirs(CIKTI, exist_ok=True)assert os.path.exists(BIRIM_YOL), (    BIRIM_YOL + " YOK. Colab korpus URETMEZ -- once yerelde "    "birim_14.kur(ayar, onbellek=<yol>) ile uretip Drive'a koyun.")COMMIT = subprocess.run(["git", "-C", KOD, "rev-parse", "--short", "HEAD"],                        capture_output=True, text=True).stdout.strip()print("commit", COMMIT, " birim %.1f MB" % (os.path.getsize(BIRIM_YOL) / 1e6))print("cikti  ", CIKTI)

In [ ]:
# 2 VERI + SINAV  |  CPU  |  ARKA PLAN# veri_kur grafi TOHUMDAN kuruyor (~1 sn), Drive'a gitmiyor.# Drive'dan gelen tek sey birim akisi.KURULUM, KLOG = "kosuyor", []T_KUR = [0.0, 0.0]def _kur():    global KURULUM    T_KUR[0] = time.time()    try:        g = globals()        import ayar_14 as AY, taban_14 as MT, veri_14 as V14        import birim_14 as BR, olcme_14 as OL, model_14 as M14        g.update(AY=AY, MT=MT, V14=V14, BR=BR, OL=OL, M14=M14)        A = AY.AYAR        v = MT.veri_kur(A, yaz=lambda *a, **k: None)        L = MT.olcme_listeleri(A, v)        assert V14.IZ == AY.IZ_GRAF, V14.IZ        assert MT.olcme_izi(L) == AY.IZ_OLCME, MT.olcme_izi(L)        KLOG.append("izler TUTUYOR  graf %s  olcme %s" % (V14.IZ, MT.olcme_izi(L)))        KLOG.append("bolmeler  " + "  ".join(            "%s:%d" % (k, len(x)) for k, x in L.items() if hasattr(x, "__len__")))        b = BR.yukle(BIRIM_YOL, yaz=KLOG.append)        KLOG.append(BR.kapi(b))        G = V14.kur(A.veri_tohum)        E_ad = [x for t in V14.TIPLER for x in G["ad"][t]]        g.update(v=v, L=L, b=b, E_ad=E_ad)        KURULUM = "bitti"    except Exception as e:        import traceback        KURULUM = "HATA: %s: %s" % (type(e).__name__, e)        KLOG.append(traceback.format_exc()[-700:])    T_KUR[1] = time.time()threading.Thread(target=_kur, daemon=True).start()print("kurulum ARKA PLANDA basladi -- durum icin R")

In [ ]:
# 3 EGITIM  |  GPU  |  ARKA PLANEGITIM, ELOG, MDL = "kosuyor", [], NoneEP = [0, 0]                      # [bitmis epok, toplam]T_EG = [0.0, 0.0]def _egit():    global EGITIM, MDL    T_EG[0] = time.time()    try:        assert KURULUM == "bitti", "once 2 bitsin (durum: %s)" % KURULUM        A = AY.AYAR        torch.manual_seed(A.tohum)        m = M14.Yol(len(b), D=AY.D_DURUM, d=AY.D_OKUMA, K=AY.K_KOD,                    tam=M14.sinif_ayir(b.say, AY.K_TAM), saat=AY.SAAT).to("cuda")        ELOG.append(M14.kapi(m).replace("\n", "\n  "))        P = torch.as_tensor(b.pencere(AY.PENCERE).copy(), device="cuda")        op = torch.optim.Adam(m.parameters(), lr=AY.LR)        gg = torch.Generator(device="cuda").manual_seed(A.tohum)        N, BS = len(P), AY.BATCH        EP[1] = AY.EPOK        ELOG.append("pencere %s  adim/epok %d  batch %d"                    % (tuple(P.shape), (N + BS - 1) // BS, BS))        for ep in range(1, AY.EPOK + 1):            perm = torch.randperm(N, device="cuda", generator=gg)            tot = ns = 0.0            for i in range(0, N, BS):                X = P[perm[i:i + BS]]                k, uye = m.kayip(X, AY.A1_DIS, AY.A2_CAPA, AY.A3_DUZEN,                                 AY.DELTA, AY.BETA, AY.R_CAPA)                op.zero_grad(set_to_none=True)                k.backward()                op.step()                tot += float(uye) * len(X)                ns += len(X)            EP[0] = ep            ELOG.append("  epok %d/%d  uye %.4f  (%.0f sn)"                        % (ep, AY.EPOK, tot / ns, time.time() - T_EG[0]))        MDL = m        EGITIM = "bitti"    except Exception as e:        import traceback        EGITIM = "HATA: %s: %s" % (type(e).__name__, e)        ELOG.append(traceback.format_exc()[-700:])    T_EG[1] = time.time()threading.Thread(target=_egit, daemon=True).start()print("egitim ARKA PLANDA basladi -- durum icin R")

In [ ]:
# 4 DOGRULUK + SONUC KAYDI  |  GPU  |  ARKA PLAN# Biter bitmez Drive'a yazar: Colab dusse de sonuc DURUR.OLCUM, OLOG, SONUC = "kosuyor", [], NoneT_OL = [0.0, 0.0]def _olc():    global OLCUM, SONUC    T_OL[0] = time.time()    try:        assert EGITIM == "bitti", "once 3 bitsin (durum: %s)" % EGITIM        soru_tip = {}        for i, t in enumerate(V14.TIPLER):            sz = {"KISI": "kim", "SEHIR": "neresi", "BOLGE": "neresi"}.get(t)            soru_tip[i] = b.ix.get(sz, b.ix["hangisi"])        sn = OL.Sinav(v, E_ad, list(V14.ILISKI), V14.TR, b.kok, b.ix,                      b.korunan, soru_tip)        sor = {}        for ad, adim in (("one", 1), ("seen", 2), ("comp", 2), ("ent", 2)):            q, at = sn.bolme(L.get(ad) or [], adim)            sor[ad] = q            OLOG.append("  %-5s %5d soru  (%d kurulamadi)" % (ad, len(q), at))        OLOG.append(OL.soru_kapisi(sor["one"], b.dizi))        bitis = set(b.ix[x] for x in ".?!" if x in b.ix)        ek_ix = set(i for i, a in enumerate(b.ad) if a.startswith("-"))        var_ix = set(w for ad in E_ad for w in sn.birim(ad))        bc = OL.Bicim(b.dizi, var_ix, bitis, ek_ix, n=len(b))        SONUC = OL.dogruluk(MDL, bc, sor, bitis,                            OL.tip_haritasi(v, E_ad, sn), r=AY.R_CAPA)        OL.yaz(SONUC, OLOG.append)        # --- SONUC KAYDI. Colab dusse de kalir.        kayit = {"commit": COMMIT, "zaman": time.strftime("%Y-%m-%d %H:%M"),                 "iz_graf": V14.IZ, "iz_birim": BR.iz(b),                 "ayar": {k: getattr(AY, k) for k in                          ("D_DURUM", "D_OKUMA", "K_KOD", "K_TAM", "SAAT",                           "R_CAPA", "DELTA", "A1_DIS", "A2_CAPA", "A3_DUZEN",                           "BETA", "PENCERE", "LR", "BATCH", "EPOK")},                 "parametre": M14.n_par(MDL), "birim": len(b),                 "egitim_log": ELOG, "dogruluk": SONUC}        with open(CIKTI + "/dogruluk.json", "w", encoding="utf-8") as f:            json.dump(kayit, f, ensure_ascii=False, indent=1)        torch.save({"model": MDL.state_dict(), "ayar": kayit["ayar"]},                   CIKTI + "/model_t0.pt")        OLOG.append("KAYDEDILDI  " + CIKTI + "/{dogruluk.json, model_t0.pt}")        OLCUM = "bitti"    except Exception as e:        import traceback        OLCUM = "HATA: %s: %s" % (type(e).__name__, e)        OLOG.append(traceback.format_exc()[-900:])    T_OL[1] = time.time()threading.Thread(target=_olc, daemon=True).start()print("olcum ARKA PLANDA basladi -- durum icin R")

In [ ]:
# R RAPOR  |  CPU  |  ANINDA doner  |  HER ZAMAN GUVENLI# Uc kosunun da durumunu ve TAM logunu basar. GPU'ya dokunmaz, hicbir# sey baslatmaz -- istenildigi kadar kosulabilir.def _sure(t):    if not t[0]:        return "-"    son = t[1] or time.time()    return "%.0f sn%s" % (son - t[0], "" if t[1] else " (suruyor)")print("=" * 62)print("model_14  commit %s   %s" % (COMMIT, time.strftime("%H:%M:%S")))print("=" * 62)for ad, dur, t in (("KURULUM", KURULUM, T_KUR), ("EGITIM", EGITIM, T_EG),                   ("OLCUM", OLCUM, T_OL)):    print("  %-9s %-34s %s" % (ad, dur, _sure(t)))if EP[1]:    kalan = ""    if EP[0] and EGITIM == "kosuyor":        bir = (time.time() - T_EG[0]) / EP[0]        kalan = "   kalan ~%.0f sn" % (bir * (EP[1] - EP[0]))    print("  epok      %d/%d%s" % (EP[0], EP[1], kalan))if torch.cuda.is_available():    print("  GPU       %.1f / %.1f GB kullanimda"          % ((torch.cuda.mem_get_info()[1] - torch.cuda.mem_get_info()[0]) / 1e9,             torch.cuda.mem_get_info()[1] / 1e9))for baslik, log in (("KURULUM", KLOG), ("EGITIM", ELOG), ("OLCUM", OLOG)):    if log:        print("\n--- " + baslik + " ---")        for s in log:            print(s if s.startswith(" ") else "  " + s)

In [ ]:
# 5 NABIZ  |  CPU  |  ARKA PLAN  |  oturum dusmesin# Uzun kosularda Colab oturumu dusmesin diye. Her turda uc kosunun da# durumunu basiyor, boylece izleme icin ayrica bir sey kosturmak# gerekmiyor (CLAUDE.md kural 8).NABIZ = Truedef _nabiz():    while NABIZ:        print("[%s] kurulum=%s  egitim=%s (%d/%d)  olcum=%s"              % (time.strftime("%H:%M:%S"), KURULUM, EGITIM, EP[0], EP[1],                 OLCUM), flush=True)        time.sleep(180)threading.Thread(target=_nabiz, daemon=True).start()print("nabiz basladi (3 dk) -- durdurmak icin NABIZ = False")